# 股票分析軟體 - Jupyter Notebook 示範

本 Notebook 展示如何使用 `stock_analyzer` 套件進行股票分析。

功能包含：
- 股價查詢
- 技術指標計算（MA、RSI、MACD、布林通道）
- 基本面資料查詢
- 投資組合分析
- 選股篩選
- 圖表繪製
- 資料匯出

In [ ]:
# 載入套件
import pandas as pd
import matplotlib.pyplot as plt

from stock_analyzer import (
    fetch_stock_data,
    fetch_stock_info,
    analyze_all_indicators,
    generate_signals,
    fetch_fundamental_data,
    format_fundamental_summary,
    Portfolio,
    StockScreener,
    plot_stock_chart,
    plot_portfolio,
    export_to_csv,
    export_to_excel,
)

# 讓圖表直接顯示在 Notebook 中
%matplotlib inline

## 1. 股價查詢

支援多市場：
- 美股：`AAPL`、`TSLA`、`MSFT`
- 台股：`2330`、`2317`（自動加上 `.TW`）
- 港股：`0700`、`3690`（自動加上 `.HK`）

In [ ]:
# 查詢蘋果公司最近一年的股價
df = fetch_stock_data("AAPL", period="1y", interval="1d")
df.tail(10)

In [ ]:
# 查詢台積電股價（台股）
df_tw = fetch_stock_data("2330", period="6mo", market="tw")
df_tw.tail()

## 2. 技術指標分析

In [ ]:
# 計算所有技術指標
df_ind = analyze_all_indicators(df)

# 查看最新訊號
signals = generate_signals(df_ind)
print(f"日期: {signals['date']}")
print(f"收盤價: {signals['close']:.2f}")
print(f"RSI: {signals['rsi']:.2f}")
print(f"MACD: {signals['macd']:.4f}")
print(f"訊號: {signals['signals']}")

# 顯示部分指標
cols = ["close", "ma_5", "ma_20", "ma_60", "rsi", "macd", "bb_upper", "bb_lower"]
df_ind[cols].tail(10)

In [ ]:
# 繪製技術分析圖
fig = plot_stock_chart(df_ind, "AAPL", figsize=(14, 10))

## 3. 基本面分析

In [ ]:
# 取得基本面資料
fund_data = fetch_fundamental_data("AAPL")
summary = format_fundamental_summary(fund_data)
summary

## 4. 投資組合分析

In [ ]:
# 建立投資組合
portfolio = Portfolio(name="示範投資組合")

# 添加庫存：代碼、股數、成本
portfolio.add_position("AAPL", shares=10, cost_basis=150)
portfolio.add_position("TSLA", shares=5, cost_basis=200)
portfolio.add_position("MSFT", shares=8, cost_basis=300)

# 查看摘要
portfolio.get_summary()

In [ ]:
# 繪製投資組合圖表
fig = plot_portfolio(portfolio, kind="both", figsize=(14, 6))

## 5. 選股篩選

In [ ]:
# 建立篩選器
screener = StockScreener()

# 條件：RSI 在 30~50 之間，且股價在 20 日均線之上
screener.add_condition("rsi", min_val=30, max_val=50)
screener.add_condition("close", operator=">", field="ma_20")

# 對股票列表進行篩選
symbols = ["AAPL", "MSFT", "GOOGL", "TSLA", "AMZN"]
results = screener.screen(symbols, period="1y")
results

## 6. 資料匯出

In [ ]:
# 匯出單檔股票資料到 CSV
csv_path = export_to_csv(df_ind, filename="AAPL_analysis.csv")
print(f"CSV 匯出路徑: {csv_path}")

In [ ]:
# 匯出多個 DataFrame 到同一個 Excel 檔案的不同工作表
excel_path = export_to_excel(
    {
        "AAPL_Price": df,
        "AAPL_Indicators": df_ind,
        "AAPL_Fundamental": summary,
        "Portfolio": portfolio.get_summary(),
    },
    filename="stock_report.xlsx",
)
print(f"Excel 匯出路徑: {excel_path}")

## 小結

以上示範涵蓋了本股票分析軟體的主要功能。你可以根據自己的需求：
1. 修改股票代碼與市場
2. 調整技術指標參數
3. 新增更多篩選條件
4. 擴充自己的投資組合策略

所有資料僅供參考，不構成投資建議。